# Error Analysis


## Setup & Imports

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"          # global (frozen prompts)

# UNI-88: point at one experiment folder. Paste the name printed by pipeline.ipynb §2.
EXPERIMENT_NAME = "experiment_test_9385_20260609_1013"       # <-- set to the run you are analysing
EXPERIMENT_DIR  = ROOT / "data" / "experiments" / EXPERIMENT_NAME
LLAMA_RUNS  = EXPERIMENT_DIR / "llama_runs"
EXPERIMENT  = EXPERIMENT_DIR / "experiment.jsonl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
print(f"ROOT = {ROOT}")

### Analysis 1 — Structural confusion: do wrong retrievals share a plot skeleton?

**Goal:** When the structural model retrieves the wrong story, is it because of a random error/noise, or whether the errors are meaningful where these "wrong" stories are structurally similar (e.g., share same archetypical event trigger and/or event type skeleton)? 

**Method:**
1. Take every query whose top-1 retrieval belongs to a different work, so the actual errors hapenned. Call the wrongly retrieved summary "wrong match". 
2. For each (query, wrong match) pair, measure how much narrative structure the two share, in four ways: 
    1. Shared *event types* via Traversky Index overlap: How much *event types* overlap between query and the wrong match?
    2. Longest in-order common event type sequence (namely, LCS ratio) for *event types*: What is the longest sequence of *event types* that appear in both the query and the wrong match? 
    3. Shared *event triggers* via Traversky Index overlap: How much *event triggers* overlap between query and the wrong match?
    4. Longest in-order common event type sequence (namely, LCS ratio) for *event triggers*: What is the longest sequence of *event triggers* that appear in both the query and the wrong match?
3. Build a chance baseline: pair the same queries with a random story (also a different work) and compute the same four measures. These are the ratios created, as if we would do things randomly.
4. Compare error pairs vs random pairs (one-sided Mann–Whitney U): if the scores of match between (query, wrong match) is higher than the scores of match (query, random match), the errors are systematic structural confusion (i.e., signal)
5. Visually inspect the top pairs by hand: the shared event sequence (the candidate plot skeleton), genres, languages, and the two texts side by side.

**Results:**
- 77.5% of queries (4,393 / 5,666) retrieve a wrong story at rank 1 (Qwen3 × events_only).
- The wrong story shares **almost twice** the event categories with the query that a random story would (Dice 0.44 vs 0.23).
- The same holds for **order**: the shared in-order event sequence is ~2× longer than chance (0.20 vs 0.11) — so it's plot *shape*, not just shared ingredients.
- Even on **exact trigger verbs**, the wrong match is ~3× above chance (0.10 vs 0.04) — so the effect is not an artifact of coarse or noisy event-type labels.
- All four differences are significant at p ≈ 0. Sanity check: the random baseline for event-type overlap (0.23) reproduces the corpus-wide mean from pipeline §8.8 / Table 5 (0.237).
- Top confusions share interpretable skeletons across languages and genres — e.g. *Sending → Arriving → Giving → Warning* shared by a German doctor drama and an Italian mafia film with **zero** shared genres.

**Interpretation:** the model does exactly what it was built to do — match plot structure. But plot skeletons are not unique to a work: many unrelated stories are built on the same shape, so structure alone cannot separate "another telling of this story" from "a different story with the same shape". That is the mechanism behind the low structural P@1, reported as error analysis in Results and reframed as evidence of prototypical, cross-cultural narrative structures (future-research pointer) in the Discussion.

In [19]:
import random

import numpy as np
from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu

# Set up encoder and condition. Although all conditions have the same set of events, the retrieval results, therefore the wrong match, changes across conditions. 
#   VIEW (embedder): qwen3_emb_0p6b | e5_mistral
#   COND (representation): events_only | temporal | causal | temporal_causal_independent | temporal_causal_joint | raw_text
VIEW, COND = "qwen3_emb_0p6b", "events_only"
SEED = 0
# Download the data from cache, instead of re-loading the full data in each run to save time
META_CACHE   = EXPERIMENT_DIR / "error_analysis_meta.jsonl"
# Get the similarities across all conditions and embedders
SIMILARITIES = EXPERIMENT_DIR / "similarities.npz"

# 1. Get per-row metadata only, since experiment.jsonl carries embeddings (~4.4 GB)
if not META_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(META_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            out.write(json.dumps({
                "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
                "lang": r.get("lang"), "genres": r.get("genres") or [],
                "triggers": [e["trigger"].lower() for e in r.get("events") or []],
                "types":    [e["event_type"]      for e in r.get("events") or []],
                "text": r.get("text", ""),
            }, ensure_ascii=False) + "\n")
meta  = [json.loads(l) for l in META_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
works = np.array([m["wikidata_id"] for m in meta])
N     = len(meta)

# 2. Rank-1 neighbor per query (similarities.npz rows follow experiment.jsonl order; diagonal is -inf)
sim = np.load(SIMILARITIES)[f"{VIEW}__{COND}"]
assert sim.shape == (N, N), f"similarities {sim.shape} vs meta rows {N} — cache stale? delete {META_CACHE}"
rank1 = sim.argmax(axis=1)
wrong = np.flatnonzero(works[rank1] != works)
error_pairs = [(int(q), int(rank1[q])) for q in wrong]
print(f"{VIEW}__{COND}: wrong story at rank 1 for {len(wrong)}/{N} queries ({len(wrong)/N:.1%})")

# 3. Skeleton-overlap measures. Inventory = set-based Dice; order = LCS ratio with the same normalization (2·LCS/(|a|+|b|)), so inventory and order numbers are directly comparable.
def dice(a, b):
    A, B = set(a), set(b)
    return 2 * len(A & B) / (len(A) + len(B)) if (A or B) else 0.0

def lcs_table(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            L[i + 1][j + 1] = L[i][j] + 1 if x == y else max(L[i][j + 1], L[i + 1][j])
    return L

def lcs_ratio(a, b):
    return 2 * lcs_table(a, b)[-1][-1] / (len(a) + len(b)) if (a or b) else 0.0

# Calculate the measures
MEASURES = {
    "Event-type inventory overlap": lambda u, v: dice(u["types"], v["types"]),
    "Event-type order overlap":     lambda u, v: lcs_ratio(u["types"], v["types"]),
    "Trigger inventory overlap":    lambda u, v: dice(u["triggers"], v["triggers"]),
    "Trigger order overlap":        lambda u, v: lcs_ratio(u["triggers"], v["triggers"]),
}

# 4. Random (Null) Baseline: same queries, but each paired with a random non-relevant candidate instead of the model's wrong choice, basically this is what overlap looks like when the pairing carries no signal
rng = random.Random(SEED)
null_pairs = []
for q, _ in error_pairs:
    j = rng.randrange(N)
    while j == q or works[j] == works[q]:
        j = rng.randrange(N)
    null_pairs.append((q, j))

# 5. Error vs null per measure
scores, table = {}, []
for name, fn in MEASURES.items():
    err = np.array([fn(meta[q], meta[c]) for q, c in error_pairs])
    nul = np.array([fn(meta[q], meta[c]) for q, c in null_pairs])
    _, p = mannwhitneyu(err, nul, alternative="greater")
    scores[name] = (err, nul)
    table.append({"Measure": name,
                  "Error pair mean":  f"{err.mean():.4f}",
                  "Random-pair mean": f"{nul.mean():.4f}",
                  "Δ":                f"{err.mean() - nul.mean():+.4f}",
                  "p":                "< .001" if p < .001 else f"{p:.3f}"})
    
# Display the table, and a note for my thesis and other readers.
display(pd.DataFrame(table))
display(Markdown(
    f"*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under "
    f"*{VIEW} × {COND}* (embedder × representation condition; set `VIEW`/`COND` above to analyse "
    f"another cell of the ablation grid). A *random pair* is the same query paired with a randomly "
    f"drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient "
    f"(symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. "
    f"*Order overlap* respects narrative order: the longest common subsequence between the two item "
    f"sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. "
    f"Both are computed over MAVEN event types and over exact (lowercased) trigger words. "
    f"*p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs."
))

qwen3_emb_0p6b__events_only: wrong story at rank 1 for 4393/5666 queries (77.5%)


,Measure,Error pair mean,Random-pair mean,Δ,p
0,Event-type inventory overlap,0.4363,0.2282,+0.2081,< .001
1,Event-type order overlap,0.2031,0.1105,+0.0925,< .001
2,Trigger inventory overlap,0.1008,0.0359,+0.0649,< .001
3,Trigger order overlap,0.0657,0.0256,+0.0401,< .001


*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under *qwen3_emb_0p6b × events_only* (embedder × representation condition; set `VIEW`/`COND` above to analyse another cell of the ablation grid). A *random pair* is the same query paired with a randomly drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient (symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. *Order overlap* respects narrative order: the longest common subsequence between the two item sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. Both are computed over MAVEN event types and over exact (lowercased) trigger words. *p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs.

In [ ]:
from IPython.display import display, Markdown

TOP_K   = 3
PREVIEW = 800   # chars of raw text shown per summary

def lcs_seq(a, b):
    """Backtrack the DP table to recover one LCS (the shared skeleton itself)."""
    L = lcs_table(a, b)
    i, j, out = len(a), len(b), []
    while i and j:
        if a[i-1] == b[j-1]:
            out.append(a[i-1]); i -= 1; j -= 1
        elif L[i-1][j] >= L[i][j-1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]

err_lcs = scores["Event-type order overlap"][0]
order   = np.argsort(-err_lcs)

md = [f"## Top {TOP_K} structural confusions under `{VIEW}__{COND}` (by event-type order overlap)\n"]
for rank, r in enumerate(order[:TOP_K], 1):
    q, c = error_pairs[r]
    mq, mc = meta[q], meta[c]
    shared_skel   = lcs_seq(mq["types"], mc["types"])
    shared_genres = sorted(set(mq["genres"]) & set(mc["genres"]))
    md.append(
        f"### {rank}. `{mq['wikidata_id']}__{mq['summary_id']}` → `{mc['wikidata_id']}__{mc['summary_id']}`"
        f" &nbsp; (cosine similarity {sim[q, c]:.3f}, event type order overlap {err_lcs[r]:.3f}, "
        f"event type inventory {dice(mq['types'], mc['types']):.3f}, event trigger inventory overlap {dice(mq['triggers'], mc['triggers']):.3f})\n\n"
        f"- **shared event-type skeleton ({len(shared_skel)} types)**: {' → '.join(shared_skel)}\n"
        f"- **shared genres**: {', '.join(shared_genres) if shared_genres else '(none)'}"
        f" &nbsp;|&nbsp; langs: {mq['lang']} vs {mc['lang']}\n\n"
        f"- **Event Types of the Query ({len(mq['types'])})**: {' → '.join(mq['types'])}\n"
        f"- **Event Types of the Wrong Match ({len(mc['types'])})**: {' → '.join(mc['types'])}\n"
        f"- **Event Triggers of the Query:** {', '.join(mq['triggers'])}\n"
        f"- **Event Triggers of the Wrong Match**: {', '.join(mc['triggers'])}\n"
        f"- **Text of the Query**: {mq['text'][:PREVIEW]}…\n"
        f"- **Text of the Wrong Match**: {mc['text'][:PREVIEW]}…\n"
    )
display(Markdown("\n".join(md)))

## Top 3 structural confusions under `qwen3_emb_0p6b__events_only` (by event-type order overlap)

### 1. `5504130__fr` → `562586__fr` &nbsp; (cosine similarity 0.797, event type order overlap 0.727, event type inventory 0.727, event trigger inventory overlap 0.182)

- **shared event-type skeleton (4 types)**: Come_together → Departing → Becoming → Know
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs fr

- **Event Types of the Query (5)**: Come_together → Departing → Becoming → Giving → Know
- **Event Types of the Wrong Match (6)**: Destroying → Come_together → Departing → Becoming → Know → GetReady
- **Event Triggers of the Query:** meets, leave, became, gave, find
- **Event Triggers of the Wrong Match**: decimated, meets, left, turn, discover, prepares
- **Text of the Query**: A young man, Paul Harrison, the son of a wealthy British businessman living in Paris, meets a beautiful young orphan, Michelle Latour. The two teenagers leave Paris for the Camargue. Michelle became pregnant and gave birth to a baby girl. The young couple and the child lead a family life until the police find them.…
- **Text of the Wrong Match**: An army veteran, the sole survivor of a decimated American patrol, meets a young South Korean, Short Round, as well as others left behind by the war. He leads them to an unoccupied Buddhist temple, which they turn into an observation camp. But when they discover that they are in the immediate vicinity of a North Korean communist camp, the troop prepares for the possibility of a fight...…

### 2. `76940500__en` → `1029697__fr` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: en vs fr

- **Event Types of the Query (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Types of the Wrong Match (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Triggers of the Query:** found, sent, release, return, giving, threatened
- **Event Triggers of the Wrong Match**: transferred, arrives, gives, warning, placed, becomes
- **Text of the Query**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.…
- **Text of the Wrong Match**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.…

### 3. `1029697__fr` → `76940500__en` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs en

- **Event Types of the Query (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Types of the Wrong Match (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Triggers of the Query:** transferred, arrives, gives, warning, placed, becomes
- **Event Triggers of the Wrong Match**: found, sent, release, return, giving, threatened
- **Text of the Query**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.…
- **Text of the Wrong Match**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.…


## (i) Lexical Bias (Not Priority)

**Goal:** Retrieval driven by shared entities or surface overlap, not narrative structure.

Our structural representations hold only event triggers (verbs) and MAVEN types — no names — so lexical bias cannot live there. It lives in the baselines (BoW, TF-IDF, raw text), which *beat* our structural models. This cell shows the baselines win by matching shared names, not plot.

**Method:**
1. **Detect entities** — run spaCy NER over each summary; tag `PERSON`, `GPE`/`LOC` (places), `ORG`, `DATE` spans.
2. **Ablate** — for every query a baseline got right at rank 1, delete those entity tokens from its vectors and re-run retrieval.
3. **Measure** — count how many correct matches collapse. A big drop ⇒ the baseline was retrieving on shared names, not narrative.
4. **Contrast** — those same queries already failed under our name-free structural model ⇒ the baseline's edge was purely the entity shortcut structure discards.

Mirrors Hatzel & Biemann (2024): removing names drops StoryEmbed P@N 85.90 → 65.89.

In [ ]:
# Backup if Givanni's ideas are not working, taking too long. Not a priotiy. Prior work has sad this, so no need to compute yourslef. 

## (ii) Structural Alignment

**Goal:** Two summaries use different words but tell the same story, and the structural model still finds the match.

This is the good case we want. The two summaries barely share any words, so the baselines (which match on shared words) fail to connect them. But our structural model still retrieves the right story — because both summaries have the same chain of events. This is where throwing away the words and keeping only the events *helps*: the events reveal the plot that the different wording was hiding. It is the exact opposite of lexical bias: there the baseline won on shared names; here the structural model wins with no shared words at all.

**Method:**
1. **Find candidates** — queries where a structural condition (events_only / temporal / causal / joint) retrieves the right story at rank 1, but BoW/TF-IDF similarity to that story is low (ideally the lexical baseline missed it).
2. **Confirm vocabulary differs** — few shared content words between the two summaries, so the match is not coming from surface words.
3. **Show the events line up** — the two summaries share a similar event-type sequence / relation pattern; that is what carried the match.
4. **Genuine structure win** — structure succeeds exactly where surface overlap can't.

Given the aggregate result (structure *loses*), expect these cases to be rare — their count vs. the lexical-bias cases is the real story for Table 5.

In [ ]:
# In the categories level, it should wokr. This would still make sense, maybe there are there are many stories that follow similar pattern, and that's why we cannot discriminate uniquer stories. Maybe there's a prototypical narrative structure that match exactly.. 
# that's shy we might have bad results --

## (iii) Structural Mismatch

**Goal:** Adding temporal/causal relations makes two unrelated stories look alike, so the model retrieves the wrong one.

This is the bad case caused by structure itself. We have events that retrieve the right story fine. Then we add the relations (`BEFORE`, `CAUSE`, etc.) on top — and retrieval gets worse. Why? Because almost every story ends up with the same generic relation skeleton (mostly `BEFORE`/`CONTAINS`/`CAUSE` chains), so the relations make unrelated stories look similar instead of telling them apart. The relation layer adds *sameness, not signal* — and that false similarity drags the wrong story to the top.

**Method:**
1. **Find candidates** — queries where events_only retrieved the right story near the top, but adding a relation condition (temporal / causal / joint) pushed it down and put a wrong story at rank 1.
2. **Look at the wrong match** — it shares the same generic relation pattern (lots of `BEFORE`/`CONTAINS`, or `CAUSE_BEFORE`) as the query, even though the actual events differ.
3. **Show the relations are boilerplate** — relation labels are low-diversity and near-identical across unrelated stories, so they add noise, not discrimination.
4. **Structure actively hurt** — the opposite of (ii): here the relations created a false match and broke a retrieval that events alone got right.

This is the mechanism behind the negative RQ2/RQ3 result: relations are too generic to discriminate, so enriching with them degrades retrieval.

## (iv) Extraction Error

**Goal:** The events themselves are wrong — missing, noisy, or mislabeled — so the structural representation is broken *before* retrieval even starts.

Unlike (iii), the problem here isn't the relations — it's the **events** BERT+CRF pulled out of the text. If the events are junk, everything built on top (relations, linearized string, embedding) is junk too. Three kinds of bad event:
- **missing** — a real event the model never detected;
- **noisy** — a word tagged as an event that isn't a story event (e.g. the linking verbs `causes`, `makes`, `due`);
- **misclassified** — a real event given the wrong MAVEN type.

**What the pipeline already removed (so we don't re-count it here):**
- **Hallucinated relation IDs** — Llama inventing links to events that don't exist — were dropped in §4.6 and saved to `hallucinated_relations.jsonl`. The pattern: ~98% invent exactly `e(N+1)`, almost always linking the *last* real event — the model assumes everything must connect to something (UNI-60).
- **Broken / unparseable relations** — JSON parse failures and context-overflow summaries — were dropped whole in §4.5 (complete-case), with counts in `experiment.yaml`.

So the live work here is the **event-level noise that survived** because it looks valid.

**Method:**
1. **Count noisy triggers** — fraction of events whose trigger is a relational/linking verb (`causes`, `makes`, `enables`, `due`) or a subword fragment (`cing`); these are extraction artifacts, not story events (UNI-68; subword slicing was fixed in UNI-105 — verify residual ≈ 0).
2. **Spot misclassification** — sample events and check trigger vs assigned MAVEN type for obvious mismatches.
3. **Report the noise budget** — pull the hallucination rate from `hallucinated_relations.jsonl` and the parse/overflow drop counts from `experiment.yaml`, so structural noise is quantified end-to-end.
4. **Tie to retrieval** — on the worst-degrading queries (largest rank drop under structure vs baseline, UNI-103), inspect the events: is the failure explained by junk triggers rather than by the method itself?

## (v) Estimating Extraction & Relation Quality (No Gold Labels)

Tell Me Again! has no ground-truth event or temporal/causal annotations, so we cannot compute true accuracy. That is normal in NLP — we defend quality with a small-sample manual precision estimate, exactly as the MAVEN authors did for their own dataset.

**Method:**
1. Randomly sample 50 summaries from the test set.
2. Read each and mark by hand: (a) extracted events — is the trigger a real event, and is the MAVEN type right? (b) inferred relations — does each link actually make sense?
3. Report estimated event precision and estimated relation precision (% correct).

**Literature:** MAVEN validated its own labels the same way: *"One of the authors also manually examined 50 random documents. The estimated accuracies of event type annotation and event mention merging are 90.1% and 86.0% respectively"* (Wang et al., 2020). We mirror this to report an *estimated relation precision* for the LLM-inferred temporal/causal links.
